# A colorful illusion: the numbers, the figures and your own imagery

Companion notebook to the GeoAIWatch post of November 2026. [Open it in Colab](https://colab.research.google.com/github/heykuldip/geoaiwatch/blob/2026-11-near-infrared/posts/2026-11-near-infrared/notebook.ipynb).

1. **The numbers.** Every number in the post, and figures 4 and 5, from the results tables. About a minute, with no imagery.
2. **The dataset** (off by default). One DynamicEarthNet frame read from the TUM record over FTP, and figures 1 to 3 remade with the checkpoints released with this post. About 35 MB of downloads.
3. **Your own Planet scenes** (off by default). True color, false color and a water index for scenes you downloaded with your own account.

Code: MIT. Results and figures: CC BY-SA 4.0. Text: CC BY-NC-ND 4.0.

In [ ]:
import os
import sys
from pathlib import Path

REF = "2026-11-near-infrared.1"  # this notebook's release; "main" tries unreleased code

if "google.colab" in sys.modules:
    repo = Path("/content/geoaiwatch")
    if not repo.exists():
        !git clone --quiet --depth 1 --branch {REF} https://github.com/heykuldip/geoaiwatch {repo}
    %cd {repo}
    !pip install --quiet -e .

here = Path.cwd()
POST = here / "posts" / "2026-11-near-infrared"
if (here / "scripts" / "figures.py").exists():
    POST = here
ROOT = POST.parents[1]
os.environ.setdefault("DEN_HOME", str(ROOT / "data" / "DynamicEarthNet"))
for name in [m for m in sys.modules if m.split(".")[0] in ("geoaiwatch", "figures")]:
    del sys.modules[name]
# a running kernel can't see a package installed after it started, so use the source directly
sys.path[:0] = [str(ROOT / "src"), str(POST / "scripts")]

## 1. The numbers

Seeds are averaged within each test cell (a place, a date and a held-out quarter), and cells count equally within each quarter. Each row is a number as the post states it.

In [ ]:
import pandas as pd

runs = pd.read_csv(POST / "runs" / "cell_level.csv")
qa = pd.read_parquet(POST / "artifacts" / "cell_qa_status.parquet")
qa["date"] = qa.date.dt.strftime("%Y-%m-%d")
CLASSES = ["agriculture", "forest_veg", "impervious", "soil", "water"]


def cells(epoch=120):
    e = runs[runs.epoch == epoch]
    return e.groupby(["fold", "cube", "date"]).mean(numeric_only=True).reset_index()


def by_fold(df, col):
    return df.groupby("fold")[col].mean()


k = cells()
gain = by_fold(k, "d_miou")
rows = [
    ("overlap score, color only", f"{by_fold(k, 'rgb0_miou').mean():.3f}"),
    ("overlap score, with near-infrared", f"{by_fold(k, 'rgbn_miou').mean():.3f}"),
    ("gain", f"{gain.mean():+.3f}"),
    ("test quarters where it went up", f"{int((gain > 0).sum())} of 4"),
    ("test tiles where it went up", f"{int((k.d_miou > 0).sum())} of {len(k)}"),
]
for c in CLASSES:
    rows.append((f"gain for {c}", f"{by_fold(k, f'd_{c}').mean():+.3f}"))
share = by_fold(k, "d_water").mean() / (len(CLASSES) * gain.mean())
rows.append(("water's share of the gain", f"{share:.0%}"))
for ep in (20, 40, 80, 120):
    rows.append((f"gain at epoch {ep}", f"{by_fold(cells(ep), 'd_miou').mean():+.3f}"))
pooled = pd.read_csv(POST / "runs" / "fold_level.csv")
pooled = pooled[pooled.epoch == 120].groupby("fold").d_miou.mean().mean()
rows.append(("gain with pooled confusion matrices", f"{pooled:+.3f}"))
for a, b in (("agriculture", "forest_veg"), ("forest_veg", "agriculture")):
    d = by_fold(k, f"rgbn_{a}_to_{b}").mean() - by_fold(k, f"rgb0_{a}_to_{b}").mean()
    rows.append((f"change in {a} read as {b}", f"{d:+.3f}"))
m = k.merge(qa, left_on=["cube", "date", "fold"], right_on=["cube", "date", "quad"])
filled, seen = m[m.fully_synthetic_cell], m[m.fully_observed_cell]
rows += [("tiles filled in entirely", f"{len(filled)} of {len(m)} ({len(filled) / len(m):.0%})"),
         ("gain on filled-in tiles", f"{filled.d_miou.mean():+.3f}"),
         ("gain on tiles seen on their date", f"{seen.d_miou.mean():+.3f} ({len(seen)} tiles)")]
pd.DataFrame(rows, columns=["in the post", "value"])

In [ ]:
PLACES = {"6466": "Renshou", "6730": "Fengcheng", "6813": "Yangzhou", "7026": "Andong"}
dates = runs[runs.epoch == 120].groupby(["cube", "date"]).d_water.mean().reset_index()
rows = [(f"water gain, {PLACES[c[:4]]}", f"{v:+.3f}")
        for c, v in dates.groupby("cube").d_water.mean().items()]
andong = dates[dates.cube.str.startswith("7026")].sort_values("d_water")
rows += [("Andong's best date", f"{andong.d_water.max():+.3f}"),
         ("Andong's worst date", f"{andong.d_water.min():+.3f}"),
         ("months of Andong's six lowest dates", ", ".join(andong.head(6).date.str[5:7]))]
frame = k[k.cube.str.startswith("6813") & (k.date == "2018-04-01") & (k.fold == "NW")].iloc[0]
rows += [("example frame's gain, three seeds", f"{frame.d_miou:+.3f}"),
         ("tiles with a smaller gain", f"{(k.d_miou < frame.d_miou).mean():.1%}"),
         ("times the median gain", f"{frame.d_miou / k.d_miou.median():.1f}")]
pd.DataFrame(rows, columns=["in the post", "value"])

### Figures 4 and 5

Drawn by the same code as the post's figures, from the same table.

In [ ]:
import tempfile

from IPython.display import Image, display

import figures as F

F.FIGDIR = Path(tempfile.mkdtemp())
classes = F.C.frozen(POST / "artifacts")["retained_classes"]
F.fig_perclass_simple(F.primary_by_fold(runs, classes), classes)
F.fig_variability(runs)
for name in ("04_per_class_simple.png", "05_variability.png"):
    display(Image(filename=str(F.FIGDIR / name), width=700))

## 2. The dataset and figures 1 to 3 (optional)

Set `GET_DATASET = True` and run the cell. It reads one image and the place's 24 labels from inside the TUM archives, without downloading them whole, and fetches the two checkpoints from this post's release. On a CPU the overlap score can differ from the committed one in the fifth decimal, so the check here allows 0.001.

In [ ]:
GET_DATASET = False

if GET_DATASET:
    import hashlib
    import subprocess
    import urllib.request

    if "google.colab" in sys.modules:
        !pip install --quiet "segmentation-models-pytorch==0.3.4"
    release = "https://github.com/heykuldip/geoaiwatch/releases/download/2026-11-near-infrared"
    sha256 = {
        "NW_s0_rgbn_ep120.pt": "eade4cc1a6e56c46cd5acfa67f7fa7b32560e4cea522a14ecdec3a96316cabb9",
        "NW_s0_rgb0_ep120.pt": "8447a35f46eb6bbf14fec2917b3e33abd95a7de0374120362096219b48a35ee4",
    }
    F.MODELDIR.mkdir(parents=True, exist_ok=True)
    for arm in ("rgbn", "rgb0"):
        ckpt = F.MODELDIR / f"NW_s0_{arm}_ep120.pt"
        if not ckpt.exists():
            urllib.request.urlretrieve(f"{release}/{ckpt.name}", ckpt)
        digest = hashlib.sha256(ckpt.read_bytes()).hexdigest()
        assert digest == sha256[ckpt.name], \
            f"{ckpt.name}: checksum {digest} differs from the release"
    subprocess.run([sys.executable, str(POST / "scripts" / "fetch_frame.py")], check=True)
    sys.argv = ["figures.py", "--fig", "all", "--tolerance", "1e-3"]
    F.main()
    for name in ("01a_true_color.png", "01b_false_color.png", "02_predictions.png",
                 "03_disagreement.png"):
        display(Image(filename=str(F.FIGDIR / name), width=700))

## 3. Your own Planet scenes (optional)

Download scenes with `python -m geoaiwatch.planet` and your own key (see the repository's README), then set `PLANET_DIR` to the folder they are in. These are PlanetScope scenes, a different product from the daily, gap-filled Planet Fusion images in DynamicEarthNet that the post used: the panels below show your own scenes, and don't reproduce the post's results. In Colab, upload them or mount Google Drive first.

In [ ]:
PLANET_DIR = None  # for example "data/planetscope/my-order"

if PLANET_DIR:
    import matplotlib.pyplot as plt
    import numpy as np
    import rasterio

    scene = sorted(Path(PLANET_DIR).rglob("*AnalyticMS_SR*.tif"))[0]
    with rasterio.open(scene) as s:
        blue, green, red, nir = s.read().astype("float32") / 10000   # SR scaled by 10,000

    def stretch(a):
        lo, hi = np.percentile(a[a > 0], (2, 98))
        return np.clip((a - lo) / (hi - lo), 0, 1)

    water = (green - nir) / np.maximum(green + nir, 1e-6)          # NDWI: water is high
    panels = [(np.dstack([stretch(red), stretch(green), stretch(blue)]), "True color", None),
              (np.dstack([stretch(nir), stretch(red), stretch(green)]),
               "Near-infrared shown as red", None),
              (water, "Water index (NDWI)", "RdBu")]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (img, title, cmap) in zip(axes, panels):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(scene.name)
    plt.show()